## Purpose

Scientifically validate RAG improvements before full production rollout using A/B testing.

## Concepts Covered

- A/B testing methodology for RAG systems
- Statistical significance testing (Welch's t-test, bootstrap CI)
- Traffic splitting with consistent hashing
- Gradual rollout strategies
- Common failure modes and mitigation

## After Completing

You will be able to:
- Design and execute controlled experiments on RAG configurations
- Calculate statistical significance with proper p-values and confidence intervals
- Implement traffic splitting and variant assignment
- Make evidence-based deployment decisions
- Avoid common A/B testing pitfalls

## Context in Track

**Module 8.2: A/B Testing for RAG Improvements**

Builds on M8.1 (RAGAS Evaluation) by adding experimental validation of improvements before full rollout. Prerequisite for production RAG deployments that require risk mitigation.

# Module 8.2: A/B Testing for RAG Improvements

**Duration:** 38 minutes  
**Prerequisites:** Level 1 complete, M8.1 RAGAS evaluation framework

## Overview

In M8.1, you built a RAGAS evaluation system that measures RAG performance. But having evaluation metrics doesn't tell you if your changes actually improve the user experience.

**The Problem:**
- You tweak chunk size from 512 to 1024 tokens
- RAGAS faithfulness goes from 0.82 to 0.87
- But production cost doubles and latency increases 300ms
- How do you know if the change helps more users than it hurts?

**Today's Solution:**
Build an A/B testing framework that scientifically validates RAG improvements with real traffic before committing to them.

## Section 1: Introduction & Setup

### What You'll Learn
- Design controlled experiments comparing control vs treatment RAG configurations
- Implement traffic splitting logic that randomly assigns users to experiment variants
- Calculate statistical significance with proper p-values and confidence intervals
- Execute gradual rollout strategies (canary deployments, blue-green switches)
- **Important:** When NOT to use A/B testing and what alternatives exist

### Prerequisites Check
✅ Working RAG system deployed to production  
✅ RAGAS evaluation framework measuring faithfulness, answer_relevance, context_precision  
✅ Ability to track queries and responses in a database  
✅ At least 100 queries per day (minimum for meaningful experiments)

In [ ]:
# Import dependencies
import sys
sys.path.append('.')

from m8_ab_testing_rag.ab_testing import (
    ExperimentConfig,
    ExperimentManager,
    TrafficSplitter,
    ABTestingRAGPipeline,
    StatisticalAnalyzer,
    RolloutController,
    calculate_required_sample_size
)

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

print("✅ All dependencies loaded")
# Expected: ✅ All dependencies loaded

## Section 2: Database Schema & Prerequisites

Before we start, we need database tables to track experiments.

### Database Schema

```sql
CREATE TABLE experiments (
    experiment_id VARCHAR PRIMARY KEY,
    name VARCHAR NOT NULL,
    description TEXT,
    control_config JSON,
    treatment_config JSON,
    traffic_split FLOAT DEFAULT 0.5,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status VARCHAR DEFAULT 'running',
    winner VARCHAR
);

CREATE TABLE experiment_assignments (
    user_id VARCHAR,
    experiment_id VARCHAR,
    variant VARCHAR,
    assigned_at TIMESTAMP,
    PRIMARY KEY (user_id, experiment_id)
);

CREATE TABLE experiment_results (
    id SERIAL PRIMARY KEY,
    experiment_id VARCHAR,
    variant VARCHAR,
    query_id VARCHAR,
    faithfulness FLOAT,
    answer_relevance FLOAT,
    context_precision FLOAT,
    latency_ms INT,
    timestamp TIMESTAMP
);
```

**Note:** For this demo, we'll use in-memory storage. In production, run these against your database.

## Section 3: Theory Foundation - A/B Testing Explained

### Core Concept

Instead of deploying changes to everyone, you split traffic between two versions:
- **Control (A):** Current production configuration
- **Treatment (B):** Your proposed improvement

Then measure the difference and use statistics to determine if treatment is actually better or just random luck.

### How It Works

```
User Query
    ↓
Is user in experiment?
    ↓ Yes
Random assignment (or retrieve existing)
    ├─ 50% → Control: chunk_size=512
    └─ 50% → Treatment: chunk_size=1024
    ↓
Both variants processed
    ↓
RAGAS metrics collected for both
    ↓
Statistical analysis: Is treatment significantly better?
    ├─ Yes (p < 0.05) → Roll out treatment
    ├─ No (p >= 0.05) → Keep control
    └─ Not enough data → Keep running experiment
```

### Why This Matters

- **Risk mitigation:** Only 10-50% of users affected if treatment is worse
- **Evidence-based decisions:** Statistical proof, not guessing
- **Cost control:** Measure if improvements are worth cost before scaling
- **Gradual rollout:** 10% → 50% → 100% instead of big-bang deployment

## Section 4: Hands-On Implementation

### Step 1: Experiment Configuration & Management

First, define what we're testing.

In [ ]:
# Create experiment configuration
config = ExperimentConfig(
    experiment_id="exp_chunk_size_001",
    name="Test Chunk Size 1024",
    description="Testing if larger chunks improve faithfulness",
    control_config={"chunk_size": 512, "overlap": 50, "top_k": 5},
    treatment_config={"chunk_size": 1024, "overlap": 100, "top_k": 5},
    traffic_split=0.5
)

# Initialize experiment manager (no DB for demo)
manager = ExperimentManager()
exp_id = manager.create_experiment(config)

print(f"✅ Created experiment: {exp_id}")
print(f"   Control: chunk_size={config.control_config['chunk_size']}")
print(f"   Treatment: chunk_size={config.treatment_config['chunk_size']}")
print(f"   Traffic split: {config.traffic_split*100}%")

# Expected: Created experiment: exp_chunk_size_001

### Step 2: Traffic Splitting & User Assignment

Use deterministic hashing for consistent user assignment.

In [ ]:
# Initialize traffic splitter
splitter = TrafficSplitter()

# Test 50/50 split with 100 users
assignments = {"control": 0, "treatment": 0}
for i in range(100):
    user_id = f"user_{i}"
    variant = splitter.assign_variant(user_id, exp_id, traffic_split=0.5)
    assignments[variant] += 1

print(f"✅ Assignment distribution (100 users):")
print(f"   Control: {assignments['control']}")
print(f"   Treatment: {assignments['treatment']}")

# Test consistency - same user gets same variant
variant1 = splitter.assign_variant("user_42", exp_id, 0.5)
variant2 = splitter.assign_variant("user_42", exp_id, 0.5)
assert variant1 == variant2
print(f"✅ Consistency test passed: user_42 got '{variant1}' both times")

# Expected: Close to 50/50 split (e.g., 48/52)

### Step 3: RAG Pipeline Integration

Execute RAG queries with variant-specific configurations.

In [ ]:
# Initialize A/B testing RAG pipeline (no retriever/LLM for demo)
pipeline = ABTestingRAGPipeline()

# Simulate 20 queries from different users
print("🔄 Running experiment with 20 queries...")
print()

for i in range(20):
    result = pipeline.query(
        question=f"What are GDPR data retention requirements?",
        user_id=f"user_{i}",
        query_id=f"query_{i}"
    )
    
    # Print compact results
    if i < 5:  # Only show first 5 to keep output small
        print(f"Query {i}: {result['variant']:9s} | Faithfulness: {result['metrics']['faithfulness']:.3f}")

print(f"...")
print(f"✅ Completed 20 queries")
print(f"   Results stored for statistical analysis")

# Expected: ~50/50 split with slight metric differences between variants

### Step 4: Statistical Analysis Engine

Determine if treatment is significantly better using Welch's t-test.

In [ ]:
# Simulate larger experiment for meaningful analysis
print("🔄 Simulating 500 additional queries for statistical power...")
for i in range(20, 520):
    pipeline.query(
        question=f"What are GDPR requirements?",
        user_id=f"user_{i}",
        query_id=f"query_{i}"
    )

# Analyze results
analyzer = StatisticalAnalyzer()
results = analyzer.analyze_experiment(
    exp_id, 
    metric="faithfulness",
    in_memory_data=pipeline._results
)

print(f"\n📊 Statistical Analysis Results:")
print(f"{'='*50}")
print(f"Control mean:    {results.control_mean:.4f}")
print(f"Treatment mean:  {results.treatment_mean:.4f}")
print(f"Difference:      {results.difference:.4f} ({results.percent_change:+.2f}%)")
print(f"P-value:         {results.p_value:.4f}")
print(f"Significant:     {results.is_significant} (α=0.05)")
print(f"95% CI:          [{results.confidence_interval_95[0]:.4f}, {results.confidence_interval_95[1]:.4f}]")
print(f"Winner:          {results.winner}")
print(f"Sample sizes:    Control={results.sample_size_control}, Treatment={results.sample_size_treatment}")
print(f"\n💡 Recommendation:")
print(f"   {results.recommendation}")

# Expected: With simulated data, likely shows small improvement in treatment

### Step 5: Gradual Rollout Controller

Safely deploy winning variants with incremental traffic increases.

In [ ]:
# Create rollout controller
rollout = RolloutController()

# Define gradual rollout: 10% → 50% → 100%
schedule = rollout.create_rollout_schedule(
    experiment_id=exp_id,
    stages=[
        (0.1, timedelta(days=1)),   # 10% for 1 day
        (0.5, timedelta(days=2)),   # 50% for 2 days
        (1.0, timedelta(days=0))    # 100% (full rollout)
    ]
)

print("✅ Rollout schedule created:")
for i, (traffic, duration) in enumerate(schedule['stages']):
    print(f"   Stage {i+1}: {traffic*100:>5.0f}% traffic for {duration.days} day(s)")

print(f"\n💡 This allows you to:")
print(f"   - Start with 10% traffic to minimize risk")
print(f"   - Monitor metrics before increasing")
print(f"   - Rollback quickly if issues detected")

# Expected: 3-stage rollout plan displayed

## Section 5: Sample Size Calculation

Calculate required sample size BEFORE starting experiments to avoid wasting time.

In [ ]:
# Calculate required sample size for different effect sizes
print("📏 Sample Size Calculation")
print("=" * 50)

for effect_size in [0.01, 0.02, 0.03, 0.05]:
    required_n = calculate_required_sample_size(
        effect_size=effect_size,
        alpha=0.05,
        power=0.8,
        std_dev=0.1
    )
    
    # Estimate runtime at 1000 queries/day
    queries_per_day = 1000
    days_needed = (required_n * 2) / queries_per_day
    
    print(f"Effect size {effect_size:.0%} improvement:")
    print(f"  Need {required_n:>5} samples/variant ({required_n*2:>5} total)")
    print(f"  Runtime: {days_needed:.1f} days @ 1K queries/day")
    print()

print("💡 Key insight: Detecting small improvements requires LOTS of data")
print("   2% improvement → ~4 days minimum at 1K/day traffic")

# Expected: Shows sample size increases dramatically for smaller effects

## Section 6: WHEN NOT TO USE A/B Testing ⚠️

A/B testing isn't always the right choice. Here are scenarios to avoid it:

### Scenario 1: Low Traffic (<1000 queries/day)

**Why it fails:**
- At 50/50 split with 200 queries/day, you get 100 per variant
- To reach 1000+ samples per variant = 10+ days
- During those 10 days, users stuck with potentially worse control
- Market conditions change, data becomes stale

**Use instead:** Before/After comparison
- Deploy to everyone, measure for 3 days
- Make next change, measure again
- Faster iteration, acceptable tradeoff of less rigor

### Scenario 2: Testing Multiple Factors Simultaneously

**Why it fails:**
- Want to test (chunk_size: 512 vs 1024) AND (top_k: 5 vs 10) AND (temperature: 0.7 vs 0.9)
- That's 2 × 2 × 2 = 8 combinations
- Need 1000 samples × 8 = 8000 total samples
- PLUS you can't isolate which factor caused improvement

**Use instead:** Sequential experiments
- Test chunk_size first (2 days)
- Then test top_k with winning chunk_size (2 days)
- Then test temperature (2 days)
- Total: 6 days, clear causality

### Scenario 3: Change Is Obviously Better or Worse

**Why it fails:**
- New model improves faithfulness from 0.45 → 0.92 (clear win)
- New config breaks 50% of queries with errors (clear loss)
- Cost goes from $100/month → $4000/month (clearly unaffordable)

**Use instead:** Shadow mode for 24 hours
- Deploy to 10% for 1 day
- If no fires, go to 100%
- Don't wait for statistical significance

### Scenario 4: Can't Afford Worse Variant for Users

**Why it fails:**
- Medical compliance (wrong answer → patient harm)
- Financial compliance (wrong answer → regulatory violation)
- Critical security systems
- A/B testing means 50% might get worse results

**Use instead:** Shadow mode testing
- Run treatment alongside control
- Don't show treatment to users
- Validate offline before deploying

### ✅ Use A/B testing when:
- You have 1000+ queries/day
- Single-factor experiments
- Change impact is unclear (needs data)
- Can tolerate 10-50% of users potentially getting worse variant

### ❌ Avoid A/B testing when:
- Traffic <1000/day → Use Before/After
- Testing multiple factors → Use Sequential or Bandit
- Change is obviously better/worse → Use Shadow Mode
- Can't risk user impact → Use Shadow Mode

## Section 7: COMMON FAILURES 🐛

### Failure 1: Insufficient Sample Size (Underpowered Experiments)

**What happens:**
- Run experiment with only 150 samples per variant
- Get p-value = 0.3847 (not significant)
- Can't detect real 2-3% improvements with small samples

**Root cause:**
- Small samples have high variance (noise drowns out signal)
- P-value depends on sample size
- With 150 samples, can only detect huge differences (>10%)

**The fix:**
- Calculate required sample size BEFORE starting (use `calculate_required_sample_size`)
- Set `min_sample_size` in ExperimentConfig
- Don't analyze until reaching minimum
- If timeline too long, increase traffic_split to treatment (90/10 instead of 50/50)

**Prevention:**
Don't check results every day hoping for significance (leads to p-hacking)

### Failure 2: Selection Bias in Traffic Splitting

**What happens:**
- Assignment based on user_id integer (even/odd)
- Even users = newer signups (different behavior)
- Treatment appears better but it's just user differences

**Root cause:**
- User IDs aren't random (assigned sequentially at signup)
- Even/odd split creates cohort bias
- Any non-random assignment risks confounding variables

**The fix:**
- ALWAYS use cryptographic hashing (md5, sha1)
- Include experiment_id in hash
- Validate distribution after first 100 assignments
- Check for correlations: `SELECT variant, AVG(user_age_days) GROUP BY variant`

### Failure 3: Multiple Testing Problem

**What happens:**
- Running 20 experiments simultaneously at p < 0.05
- Find 3 "significant" results
- Expected false positives = 20 × 0.05 = 1

**Root cause:**
- At p < 0.05, you have 5% false positive rate
- With many tests, almost guaranteed to find false positives
- Those "winners" might all be random noise

**The fix:**
- Apply Bonferroni correction: adjusted_alpha = alpha / num_tests
- Or use Benjamini-Hochberg (less conservative)
- Limit concurrent experiments (max 3-5 at a time)
- Pre-register experiments (decide stopping criteria BEFORE looking)

### Failure 4: Premature Rollout Decisions

**What happens:**
- Check experiment after 2 days
- See p=0.04 (barely significant)
- Roll out immediately
- Week later, effect disappears

**Root cause:**
- P-value fluctuates during experiment
- Early significance often regresses to mean
- Weekend vs weekday traffic patterns differ

**The fix:**
- Wait for minimum sample size (1000+ per variant)
- Run for at least one full week (covers weekend patterns)
- Check confidence interval width (narrow = more stable)
- Use sequential testing with proper boundaries

In [ ]:
# Demonstrate Failure 1: Insufficient Sample Size
print("🐛 Demonstrating Failure 1: Insufficient Sample Size")
print("=" * 60)

# Create small experiment (only 150 samples per variant)
small_pipeline = ABTestingRAGPipeline()
small_manager = ExperimentManager()

small_config = ExperimentConfig(
    experiment_id="exp_small",
    name="Underpowered Test",
    description="Too few samples",
    control_config={"chunk_size": 512},
    treatment_config={"chunk_size": 1024},
    traffic_split=0.5
)
small_manager.create_experiment(small_config)

# Run only 300 queries (150 per variant)
for i in range(300):
    small_pipeline.query(
        question="Test query",
        user_id=f"user_{i}",
        query_id=f"query_small_{i}"
    )

# Try to analyze
small_analyzer = StatisticalAnalyzer()
small_results = small_analyzer.analyze_experiment(
    "exp_small",
    metric="faithfulness",
    in_memory_data=small_pipeline._results
)

print(f"Result: {small_results.winner} (p={small_results.p_value:.4f})")
print(f"Samples: Control={small_results.sample_size_control}, Treatment={small_results.sample_size_treatment}")
print(f"\n{small_results.recommendation}")

# Expected: "Keep running, need 1000 samples per variant"

## Section 8: DECISION CARD 🎯

**✅ BENEFIT:**
Validate RAG improvements scientifically before full rollout. Measure real user impact with statistical rigor (p<0.05 confidence). Catch regressions on 10-50% of traffic instead of 100%. Typical use: test chunk_size change, measure 2-3% faithfulness improvement, roll out only if significant.

**❌ LIMITATION:**
Requires 1000+ requests per variant for significance (typically 3-7 days at 1K/day traffic). Cannot detect small improvements (<2%) without massive sample sizes (10K+ per variant). Adds 25-50ms latency per request for variant lookup and result storage. Does not catch edge-case failures affecting <1% of queries.

**💰 COST:**
Implementation time: 8-12 hours to build framework (traffic splitting, statistical analysis, rollout logic). Monthly cost: $5-10 at 1K/day traffic (database storage), scales to $50-80 at 10K/day. Complexity: adds ~800 lines of code, 4 database tables, requires understanding of t-tests and p-values.

**🤔 USE WHEN:**
You have 1000+ daily queries, need to validate improvements before full deployment, can tolerate 10-50% of users potentially seeing degraded performance during testing, and have engineering resources to build and maintain the system. Ideal for testing single-factor changes (chunk size, top-k, temperature) with measurable impact on RAGAS metrics.

**🚫 AVOID WHEN:**
Traffic is <1000/day (use before/after comparison instead), testing multiple factors simultaneously (use sequential testing or multi-armed bandit), change impact is obviously large (use shadow mode for 24h validation then deploy), or operating in high-stakes domain where serving degraded variant is unacceptable (use shadow mode with offline analysis).

## Section 9: Visualization

Visualize experiment results to communicate findings effectively.

In [ ]:
# Visualize experiment results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Extract data from our main experiment
control_data = [r['metrics']['faithfulness'] for r in pipeline._results 
                if r['variant'] == 'control']
treatment_data = [r['metrics']['faithfulness'] for r in pipeline._results 
                  if r['variant'] == 'treatment']

# Plot 1: Distribution comparison
axes[0].hist(control_data, bins=20, alpha=0.5, label='Control', color='blue')
axes[0].hist(treatment_data, bins=20, alpha=0.5, label='Treatment', color='green')
axes[0].axvline(results.control_mean, color='blue', linestyle='--', linewidth=2, label=f'Control μ={results.control_mean:.3f}')
axes[0].axvline(results.treatment_mean, color='green', linestyle='--', linewidth=2, label=f'Treatment μ={results.treatment_mean:.3f}')
axes[0].set_xlabel('Faithfulness Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Faithfulness Scores')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Confidence interval
variants = ['Control', 'Treatment']
means = [results.control_mean, results.treatment_mean]
ci_lower = [results.control_mean, results.confidence_interval_95[0] + results.control_mean]
ci_upper = [results.control_mean, results.confidence_interval_95[1] + results.control_mean]
errors = [[means[i] - ci_lower[i] for i in range(2)], 
          [ci_upper[i] - means[i] for i in range(2)]]

axes[1].bar(variants, means, color=['blue', 'green'], alpha=0.7)
axes[1].errorbar(variants, means, yerr=errors, fmt='none', color='black', capsize=10, capthick=2)
axes[1].set_ylabel('Mean Faithfulness')
axes[1].set_title(f'Comparison with 95% CI\\n(p={results.p_value:.4f}, Winner: {results.winner})')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✅ Visualization complete")
print(f"   Clear visual evidence of difference: {results.is_significant}")

## Section 10: Summary & Next Steps

### What You Built Today

✅ Complete A/B testing framework that scientifically validates RAG improvements  
✅ Traffic splitting logic using cryptographic hashing for unbiased assignment  
✅ Statistical analysis engine with Welch's t-test and bootstrap confidence intervals  
✅ Gradual rollout controller with automatic traffic adjustment  

### Key Takeaways

1. **Always calculate sample size first** - Don't waste time on underpowered experiments
2. **Use cryptographic hashing for assignment** - Avoid selection bias
3. **Wait for minimum samples** - Don't check results daily (p-hacking)
4. **Know when NOT to use A/B testing** - Low traffic? Use before/after instead
5. **Apply corrections for multiple tests** - Bonferroni or FDR control

### Production Checklist

Before deploying to production:
- [ ] Database tables created with proper indexes
- [ ] Monitoring dashboard (Grafana) showing experiment metrics
- [ ] Automatic rollback on error rate spikes
- [ ] Documentation of experiment design and stopping criteria
- [ ] Sample size calculator for planning
- [ ] Alerts for assignment distribution skew

### Next Module

**M8.3: Continuous Monitoring & Alerting**  
Learn to detect regressions in production before users complain.

### Practice Challenges

**🟢 EASY:** Implement basic A/B test for top_k parameter  
**🟡 MEDIUM:** Add statistical significance testing and gradual rollout  
**🔴 HARD:** Production-grade with multi-metric analysis and safety checks

In [ ]:
print("=" * 70)
print(" ✅ MODULE 8.2: A/B TESTING FOR RAG IMPROVEMENTS - COMPLETE")
print("=" * 70)
print()
print("You now have the tools to:")
print("  • Scientifically validate RAG improvements before full deployment")
print("  • Minimize risk by testing on 10-50% of traffic first")
print("  • Make evidence-based decisions with statistical rigor")
print("  • Safely roll out changes with gradual traffic increases")
print()
print("Next steps:")
print("  1. Apply this to your own RAG system")
print("  2. Test chunk_size, top_k, or temperature parameters")
print("  3. Run for 3-7 days to collect sufficient data")
print("  4. Use statistical analysis to make rollout decision")
print()
print("⚠️  Remember: A/B testing requires 1000+ queries/day")
print("   If you have less, use before/after comparison instead")
print()
print("=" * 70)